# Notebook 02 · 亲手拆开 VLM 的三个零件

> 对应 **Day 2 – Day 6**。这个 notebook 不下载任何模型，
> 全部用 `src/minivlm/` 里手写的代码，**在 CPU 上几秒跑完**。
>
> 目的：把 notebook 01 里「跑通」的东西**拆开看**。
> notebook 01 让你看见「图片变成了 1369 个 token」，
> 这个 notebook 让你看见**这些 token 是怎么被算出来的、每一层的形状是怎么变的**。

---

### 三个零件

```
                    ┌──────────────────┐   ┌──────────┐   ┌────────────┐
  图片 ───────────→ │  视觉编码器      │──→│ 连接器   │──→│ 语言主干   │──→ 回答
                    │  (ViT / SigLIP)  │   │ (MLP)    │   │ (Qwen)     │
                    └──────────────────┘   └──────────┘   └────────────┘
                      把像素变成向量        把视觉向量    把 token 变成
                      保留空间结构          翻译成 LLM    下一个 token
                                            能懂的语言
```

后面 8 周你做的所有事情，本质都是在调这三个零件的关系：

| 你在做的事 | 动的是哪个零件 |
|---|---|
| LoRA 微调 | 语言主干 + **连接器**（关键！）|
| 数据合成时选图片尺寸 | 影响连接器输出的 token 数 |
| 推理量化 | 语言主干（视觉塔不能一起量化）|
| 换视觉塔 | 视觉编码器（本项目不动）|

---

### 怎么跑

```bash
cd <仓库根目录>
pip install -r requirements-core.txt
jupyter lab notebooks/02_minivlm_walkthrough.ipynb
```

**必须在仓库根目录启动** —— notebook 会 `import src.minivlm`。
如果在 `notebooks/` 下面启动会 import 失败。

## 0 · 让 notebook 能找到仓库代码

Jupyter 的当前目录不一定在仓库根目录，所以这里显式把根目录加进 `sys.path`。
这一步在很多教程里被省略，然后新手会卡在 `ModuleNotFoundError: No module named 'src'` 上很久。

In [1]:
import os, sys
from pathlib import Path

# 从当前目录往上找，直到找到含 src/minivlm 的那一层
def find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "src" / "minivlm" / "vision.py").exists():
            return cand
    return None

ROOT = find_repo_root()
if ROOT is None:
    raise SystemExit("找不到仓库根目录。请在 multimodal-lab/ 目录下启动 jupyter。")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)
print("仓库根目录:", ROOT)

import torch
torch.manual_seed(0)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("→ 这个 notebook 用 CPU 就够，不需要 GPU")

仓库根目录: /Users/wzh/WorkBuddy/2026-09-20-16-06-49/multimodal-lab
torch: 2.14.0 | CUDA: False
→ 这个 notebook 用 CPU 就够，不需要 GPU


In [2]:
from src.minivlm.vision import (
    TinyViT, PatchEmbedding, MultiHeadSelfAttention, ViTBlock,
    build_2d_sincos_pos_embed, estimate_visual_tokens, count_patches,
)
from src.minivlm.connector import build_connector, LinearConnector, MLPConnector, PerceiverResampler

print("导入成功")
print("可用计数器:", estimate_visual_tokens(1024, 1024))

导入成功
可用计数器: (1296, (72, 72))


# 零件一 · 视觉编码器

## 1.1 把图片切成 patch —— 这一步其实就是一个卷积

ViT 做的第一件事：把 H×W×3 的图片切成 N 个 14×14 的小块，
每个小块展平成一个向量。

**关键洞察**：这个操作等价于**用一个 stride=14、kernel=14 的卷积核**。
所以代码里就真的用一个 `nn.Conv2d` 实现，而不是手写切片循环 ——
又快又不容易写错。

下面验证这两种实现是否等价。

In [ ]:
IMG, PATCH, D = 224, 14, 64

# 方式 A：手写切片（直观，但慢）
x = torch.randn(1, 3, IMG, IMG)
unfolded = x.unfold(2, PATCH, PATCH).unfold(3, PATCH, PATCH)    # (1,3,16,16,14,14)
manual = unfolded.permute(0, 2, 3, 1, 4, 5).reshape(1, -1, PATCH * PATCH * 3)

# 方式 B：卷积（快，实际用这个）
proj = torch.nn.Conv2d(3, D, kernel_size=PATCH, stride=PATCH)
conv_out = proj(x)                                             # (1,D,16,16)
conv_flat = conv_out.flatten(2).transpose(1, 2)                # (1,256,D)

print("手写切片 ->", tuple(manual.shape))
print("卷积     ->", tuple(conv_flat.shape))
print(f"\n每张图的 patch 数: {(IMG//PATCH)**2}  = ({IMG}/{PATCH})^2")
print(f"（与 count_patches 函数一致: {count_patches(IMG, IMG, PATCH)}）")

# 用卷积权重去验证「等价」：把 conv 的权重 reshape 成投影矩阵，作用在手写切片上
with torch.no_grad():
    W = proj.weight.reshape(D, -1)           # (D, 3*14*14)
    same = manual @ W.T
print("\n用同一个权重矩阵作用在手写切片上，结果和卷积一致吗?",
      torch.allclose(same, conv_flat, atol=1e-5))

## 1.2 位置编码：为什么用 sincos 而不是学一个

patch 展平之后，**空间信息丢了** —— 第 5 个 patch 和第 6 个 patch 哪个在左边？
靠位置编码补回来。

两种做法：
- **可学习的位置编码**（原版 ViT）：每个位置一个向量，训出来。缺点：**换个图片尺寸就不认识了**
- **二维 sincos**（SigLIP 等）：用三角函数直接算出来，**任意尺寸都能外推**

这对我们很重要：电商图片尺寸千奇百怪，我们不能让模型只认 224×224。

In [ ]:
import matplotlib.pyplot as plt

GRID, DIM = 16, 64
pe = build_2d_sincos_pos_embed(DIM, GRID)      # (GRID*GRID, DIM)
print("位置编码形状:", tuple(pe.shape))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(pe[:, :DIM//2].numpy(), aspect="auto", cmap="RdBu")
axes[0].set_title("前一半维度 (行方向编码)")
axes[0].set_xlabel("维度"); axes[0].set_ylabel("patch 序号")

axes[1].imshow(pe[:, DIM//2:].numpy(), aspect="auto", cmap="RdBu")
axes[1].set_title("后一半维度 (列方向编码)")
axes[1].set_xlabel("维度")

# 把某个维度 reshape 回 2D，看空间结构
dim_to_show = 0
axes[2].imshow(pe[:, dim_to_show].reshape(GRID, GRID).numpy(), cmap="viridis")
axes[2].set_title(f"维度 {dim_to_show} reshape 回 {GRID}x{GRID}")

plt.tight_layout()
plt.show()

print("观察右边那张图：注意它有明显的空间周期结构。")
print("这就是「二维」的含义 —— 行和列的编码是分开算的，所以能外推到没见过的尺寸。")

## 1.3 注意力：ViTBlock 在做什么

一个 ViTBlock = **自注意力 + MLP**，都带残差连接、都用 **pre-norm**（norm 放在前面）。

`pre-norm` 为什么重要：训练深层网络时更稳，不需要 warmup 那么讲究。
现代 VLM 全都是 pre-norm，所以你在真实模型里看到的顺序是：

```
x = x + Attention(Norm(x))     ← 注意 Norm 在里面，不在外面
x = x + MLP(Norm(x))
```

而不是老式的：
```
x = Norm(x + Attention(x))
```

In [ ]:
D_MODEL, N_PATCH, N_HEAD = 64, 25, 4

attn = MultiHeadSelfAttention(D_MODEL, N_HEAD)
blk = ViTBlock(D_MODEL, N_HEAD)

x = torch.randn(2, N_PATCH, D_MODEL)
with torch.no_grad():
    y_attn = attn(x)
    y_blk = blk(x)

print(f"输入         : {tuple(x.shape)}   (batch, patch, dim)")
print(f"注意力输出   : {tuple(y_attn.shape)}   ← 形状不变（attention 是 token 之间的混合）")
print(f"ViTBlock 输出: {tuple(y_blk.shape)}   ← 形状依然不变")
print()
print("关键：**整条视觉塔里形状从头到尾不变** (B, N, D)。")
print("变化发生在两个地方：")
print("  1. patch_embed  : (B,3,H,W) -> (B,N,D)      —— 像素变 token")
print("  2. 连接器       : (B,N,D_v) -> (B,M,D_llm)  —— 跨模态翻译")
print()
print("**这两个「形状变化点」就是整个 VLM 的全部魔法。**")
print("其他所有层都只是在不改变形状的前提下加工信息。")

## 1.4 完整的 TinyViT 前向

现在把零件拼起来，看每一步的形状。

注意 `TinyViT` 的配置（12 层、384 维）远小于真实的 SigLIP-SO400M（27 层、1152 维、878M 参数），
但**结构完全一致**。这就是「能读懂的最小实现」的意思。

In [ ]:
vit = TinyViT(img_size=224, patch_size=14, embed_dim=384, depth=12, n_head=6, mlp_ratio=4.0)
vit.eval()

n_params = sum(p.numel() for p in vit.parameters())
print(f"TinyViT 参数量: {n_params/1e6:.2f} M")
print(f"（真实 SigLIP-SO400M: ~878 M，大约 {878/(n_params/1e6):.0f} 倍）")
print()

x = torch.randn(2, 3, 224, 224)
print("-" * 62)
print(f"{'阶段':<28s} {'形状':<22s}")
print("-" * 62)
print(f"{'输入图片':<28s} {str(tuple(x.shape)):<22s}")

with torch.no_grad():
    p = vit.patch_embed(x)
    print(f"{'patch_embed (卷积)':<28s} {str(tuple(p.shape)):<22s}")

    h = p + vit.pos_embed
    print(f"{'加位置编码':<28s} {str(tuple(h.shape)):<22s}")

    for i in range(2):
        h = vit.blocks[i](h)
    print(f"{'前 2 个 ViTBlock':<28s} {str(tuple(h.shape)):<22s}")

    for blk in vit.blocks[2:]:
        h = blk(h)
    print(f"{'剩余 10 个 ViTBlock':<28s} {str(tuple(h.shape)):<22s}")

    out = vit.norm(h)
    print(f"{'最终 norm':<28s} {str(tuple(out.shape)):<22s}")
print("-" * 62)
print()
print("结论：视觉塔的输入是 (B,3,H,W)，输出是 (B,N,D)。")
print(f"这里 N = {(224//14)**2}（patch 数），D = 384（隐藏维度）")

---

# 零件二 · 连接器

## 2.1 它要解决两个问题

视觉塔输出 `(B, N, 1152)`，语言模型要 `(B, L, 896)`。所以连接器必须：

1. **改维度**：1152 → 896
2. **改数量**（可选）：N 个视觉 token → M 个，M 可以小于 N

只做第 1 件事的叫 MLP；两件都做的叫 Perceiver Resampler / Q-Former。

**为什么现在主流选 MLP（不做压缩）？**

| | MLP | Perceiver Resampler |
|---|---|---|
| 参数 | 极少 | 多 |
| token 数 | N（不压缩）| M（可固定，比如 64）|
| 空间信息 | 完整保留 | **压缩时有损** |
| OCR / 细粒度 | **强** | 弱（小字容易丢）|

电商客服**极度依赖看小字**（尺码表、标签、瑕疵细节），
所以 Qwen2.5-VL 选了 MLP 路线，只在 patch 层面做 2×2 合并
（4 个 patch → 1 个 token，这是**无损的局部拼接**，不是有损的注意力压缩）。

下面把三种连接器都跑一遍，看参数量和输出 token 数的差异。

In [ ]:
D_V, D_L, B, N = 1152, 896, 2, 1024      # N = 448x448 图的 patch 数

x_vis = torch.randn(B, N, D_V)

print(f"视觉塔输出: {tuple(x_vis.shape)}   (batch, patch, D_vision={D_V})")
print(f"LLM 期待  : (batch, ?, D_llm={D_L})")
print()
print("=" * 76)
print(f"{'连接器':<22s} {'参数量':>12s} {'输出 token 数':>14s} {'输出形状':>22s}")
print("-" * 76)

results = {}
for kind, kwargs in [
    ("linear", {}),
    ("mlp", {}),
    ("perceiver", {}),
]:
    conn = build_connector(kind, D_V, D_L, **kwargs)
    conn.eval()
    with torch.no_grad():
        y = conn(x_vis)
    n_p = sum(p.numel() for p in conn.parameters())
    results[kind] = (n_p, y.shape[1])
    label = {"linear": "LinearConnector", "mlp": "MLPConnector", "perceiver": "PerceiverResampler"}[kind]
    print(f"{label:<22s} {n_p/1e6:>10.2f} M {y.shape[1]:>14d} {str(tuple(y.shape)):>22s}")
print("=" * 76)

## 2.2 关键结论：MLP 参数少到可以忽略

看上面的参数量对比 —— MLP 连接器的参数量相对整个模型（3B）来说**微不足道**。

但它**必须训**。这是多模态微调里最反直觉、也最容易漏掉的一点：

> 预训练时，连接器学到的是「把通用图片映射到语言空间」
> —— 它会区分「猫/狗/汽车」。
>
> 但**电商图是完全不同的分布**：白底商品图、局部特写、瑕疵照、
> 尺码表截图、快递面单……这些图的「视觉显著区域」和自然图片完全不同。
>
> 只训 LLM 不训连接器，模型会**看错东西** —— 而且这种错误很隐蔽，
> 因为模型会用流利的客服话术把错误包装得像对的。

所以 `configs/sft_lora_3b.yaml` 里：

```yaml
lora_include_connector: true    # ← 这一行不能改成 false
lora_include_vision: false      # ← 视觉塔冻结，但连接器要训
```

对应的 target module 是 `merger.mlp.0` 和 `merger.mlp.2`
（见 `src/train/lora_utils.py` 的 `TARGET_CONNECTOR`）。

In [ ]:
from src.train.lora_utils import TARGET_STANDARD, TARGET_CONNECTOR, FORBIDDEN, resolve_target_modules

print("标准 LLM 注意力/MLP 的 LoRA target:")
for t in TARGET_STANDARD:
    print("   ", t)
print()
print("多模态专有 —— 连接器（必须加）:")
for t in TARGET_CONNECTOR:
    print("   ", t)
print()
print("禁止 LoRA 的模块（视觉塔）:")
for t in FORBIDDEN:
    print("   ", t)
print()
print("→ Day 14 你会亲手验证：LoRA 的 B 矩阵零初始化时，")
print("  加不加 LoRA，模型输出完全一致（保证训练从「原模型」出发）。")

---

# 零件三 · 拼接：把视觉 token 塞进文本序列

## 3.1 这是整个 VLM 里最容易写错的一段代码

流程：

```
1. 文本先过 embedding 表        -> (B, L, D)   其中某些位置是 image_token
2. 找到所有 image_token 的位置
3. 在第一个位置插入 N 个视觉向量，删掉其余占位符
4. 结果变成 (B, L-1+N, D)
5. 送进 LLM
```

**为什么容易错**：多个占位符的情况下，删除位置会产生索引偏移，
用 `index_put_` 会报 `shape mismatch`。这就是那个经典的 VLM 报错。

下面用纯张量把这段逻辑走一遍，看清每个中间形状。

In [ ]:
torch.manual_seed(0)

D_L = 8          # 为了打印好看，用很小的维度
N_VIS = 6        # 假设这张图产生了 6 个视觉 token

# 模拟文本 embedding：用一个特殊值代表 image_token 的位置
IMG_TOKEN = -999.0
seq = torch.randn(6, D_L)
seq[2] = IMG_TOKEN      # 占位符在位置 2
seq = seq.unsqueeze(0)  # (1, 6, 8)
print("文本 embedding:", tuple(seq.shape))

# 模拟连接器输出
vis = torch.randn(1, N_VIS, D_L)
print("连接器输出    :", tuple(vis.shape))

# --- 拼接 ---
B, L, _ = seq.shape
pos = (seq[0, :, 0] == IMG_TOKEN).nonzero(as_tuple=True)[0]
print("\n找到占位符位置:", pos.tolist())

pieces = []
cur = 0
for p in pos.tolist():
    pieces.append(seq[:, cur:p, :])
    pieces.append(vis.to(seq.dtype))
    cur = p + 1
pieces.append(seq[:, cur:, :])

merged = torch.cat(pieces, dim=1)
print(f"\n拼接后        : {tuple(merged.shape)}")
print(f"计算验证      : {L} 个文本位置 - {len(pos)} 个占位符 + {N_VIS} 个视觉 = {L - len(pos) + N_VIS}")
print(f"形状匹配      : {merged.shape[1] == L - len(pos) + N_VIS}")
print()
print("→ 序列变长了！这就是「视觉 token 很贵」的直接体现：")
print(f"   本来 {L} 个位置，现在 {merged.shape[1]} 个。")

## 3.2 在真实的 MiniVLM 里做同一件事

`src/minivlm/model.py` 里的 `merge_visual_embeds_expand` 就是上面那段逻辑，
只不过加了错误检查 —— 当视觉 token 数和占位符数量对不上时，
它会给出**可操作的报错**，而不是一个裸的 `RuntimeError: shape mismatch`。

下面用真实的 MiniVLM 类构造一个（不需要下载任何权重，我们只测「拼接」这一步）。

In [ ]:
import inspect
from src.minivlm.model import MiniVLM

# 打印这个方法的源码 —— 这就是 Day 5-6 要读的那一段
src_code = inspect.getsource(MiniVLM.merge_visual_embeds_expand)
print(src_code)

## 3.3 上游是怎么保证「数量对得上」的？

你可能会问：为什么会有 `shape mismatch`？视觉 token 数不是算出来的吗？

**因为有两套计算视觉 token 数的地方，它们必须一致**：

| 谁在算 | 在哪算 |
|---|---|
| processor（生成 `<\|image_pad\|>` 占位符）| 图像预处理阶段，按 smart_resize 后的尺寸 |
| 视觉塔 + 连接器（真的产出 N 个向量）| 前向阶段，按实际 patch 数 |

如果两边算法有一点差异（比如一边用 `ceil` 一边用 `round`，
或者一边对齐到 28 一边对齐到 14），数量就对不上了。

**这就是 `make day4` 那个验收测试的意义** —— 它不是形式主义，
它是这条路上最高频的一个 bug。

In [ ]:
print("不同对齐算法会导致不同的 token 数 —— 演示为什么必须统一：\n")
print(f"{'尺寸':>10s} {'round 对齐':>12s} {'ceil 对齐':>12s} {'差':>6s}")
print("-" * 46)
for s in [224, 300, 448, 700, 1024, 1280]:
    n_round = (round(s/14)*14 // 14)**2 // 4
    import math
    n_ceil = (math.ceil(s/14) // 2 * 2)**2 // 4
    flag = "  <-- 对不上！" if n_round != n_ceil else ""
    print(f"{s:>10d} {n_round:>12d} {n_ceil:>12d} {n_ceil-n_round:>6d}{flag}")
print()
print("所以：**必须用官方 processor 算占位符数量**，不要自己写一套。")
print("自己写只用于「估算成本」和「理解原理」，不用于生产。")
print()
print("→ 这就是 notebook 01 里 `check_against_official` 那一步在干的事：")
print("   我们算一遍，官方 processor 算一遍，两个数必须一致。")

In [ ]:
print("=" * 66)
print("  视觉 token 成本表（2x2 merge 之后）")
print("=" * 66)
print(f"{'图片尺寸':>14s} {'patch 数':>10s} {'视觉 token':>12s} {'≈汉字':>8s}")
print("-" * 66)
for s in [224, 336, 448, 672, 896, 1024, 1280, 1536]:
    n, grid = estimate_visual_tokens(s, s)
    print(f"{str(s)+'x'+str(s):>14s} {grid[0]*grid[1]:>10d} {n:>12d} {n:>7d}字")
print("=" * 66)
print()
print("记住这张表。W2 合成数据时图片渲染多大、W3 训练时 max_length 设多少、")
print("W5 推理时 max_pixels 定多少 —— 全都从这张表推出来。")

---

# 收尾 · Day 6 的白板测试

## 合上电脑，画出这张图

每一步都标上形状。画不出来就回到对应的小节：

```
图片 (B, 3, 448, 448)
   │
   ├─ patch_embed（= 卷积）────────→ (B, ___, ___)        ← 1.1 节
   │
   ├─ + 位置编码 ──────────────────→ (B, ___, ___)        ← 1.2 节
   │
   ├─ 12 × ViTBlock（形状不变）──────→ (B, ___, ___)        ← 1.3 节
   │
   └─ 连接器 ─────────────────────→ (B, ___, ___)        ← 2.1 节

文本 → tokenizer → embedding ──────→ (B, ___, ___)        ← 有占位符
        │
        └─ 替换占位符 ─────────────→ (B, ___, ___)        ← 3.1 节

                    ↓
              [ LLM 主干 ]
                    ↓
              下一个 token
```

## 必答的三个问题

1. **哪一步是「像素→token」？哪一步是「跨模态翻译」？**
2. **为什么整条视觉塔的形状从头到尾不变？变的只有哪两个地方？**
3. **连接器为什么必须训？**（提示：想想白底商品图和自然图片的区别）

## 打卡

在 `progress/daily-log.md` 里填 Day 2–6，至少写清：
- 哪个位置让你第一次「哦，原来是这样」
- 哪里还没完全想明白（**写下来，这是下周的起点**）

## 下一步

**W2 开始**：`docs/05-data-engineering.md`。

从架构转到数据。心态上的转变：

> W1 你在理解「模型怎么工作」。
> W2 起你要开始面对一个更朴素也更重要的问题：
> **「我给它看什么？」**
>
> 8 周下来你会发现的真相是：模型结构的差别是几个点，
> 数据质量的差别是几十个点。
